# 🐛 Bug Prediction System
A complete ML pipeline to predict whether a code commit is likely to introduce a bug.

**Steps covered:**
1. Install dependencies
2. Import libraries
3. Load / generate dataset
4. Exploratory Data Analysis (EDA)
5. Feature engineering
6. Preprocessing pipeline
7. Train / test split
8. Train model (Random Forest + XGBoost)
9. Evaluate model
10. Save & reload model
11. Predict on new data

## Step 1 — Install dependencies

In [ ]:
# Run once to install required packages
!pip install pandas numpy scikit-learn xgboost matplotlib seaborn joblib --quiet

## Step 2 — Import libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, accuracy_score
)
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
import xgboost as xgb
import joblib

# Plotting style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
print('✅ All libraries imported successfully!')

## Step 3 — Load / generate dataset

We simulate a realistic code-commit dataset. Replace `generate_dataset()` with your own CSV loader if you have real data.

In [ ]:
np.random.seed(42)
N = 2000  # number of commits

def generate_dataset(n):
    """Simulate a commit-level bug prediction dataset."""
    data = {
        # Code churn features
        'lines_added':          np.random.randint(1, 500, n),
        'lines_deleted':        np.random.randint(0, 300, n),
        'files_changed':        np.random.randint(1, 20, n),
        
        # Complexity features
        'cyclomatic_complexity': np.random.randint(1, 50, n),
        'num_functions':         np.random.randint(1, 30, n),
        'avg_function_length':   np.random.uniform(5, 100, n),
        
        # Developer / history features
        'developer_experience':  np.random.randint(0, 15, n),   # years
        'prior_bugs_author':     np.random.randint(0, 50, n),
        'commit_hour':           np.random.randint(0, 24, n),
        'is_weekend':            np.random.randint(0, 2, n),
        
        # Review features
        'num_reviewers':         np.random.randint(0, 5, n),
        'review_comments':       np.random.randint(0, 30, n),
        'time_to_review_hours':  np.random.uniform(0, 72, n),
        
        # Test features
        'test_coverage_pct':     np.random.uniform(0, 100, n),
        'num_test_cases':        np.random.randint(0, 100, n),
    }
    df = pd.DataFrame(data)
    
    # Create a realistic (non-random) bug label
    bug_score = (
        0.3 * (df['lines_added'] / 500) +
        0.2 * (df['cyclomatic_complexity'] / 50) +
        0.15 * (df['prior_bugs_author'] / 50) +
        0.1 * (1 - df['test_coverage_pct'] / 100) +
        0.1 * (df['is_weekend']) +
        0.15 * (1 - df['developer_experience'] / 15)
    )
    bug_score += np.random.normal(0, 0.1, n)  # add noise
    df['is_buggy'] = (bug_score > 0.45).astype(int)
    return df

df = generate_dataset(N)

# --- OR load your own CSV ---
# df = pd.read_csv('your_dataset.csv')

print(f'Dataset shape: {df.shape}')
print(f'\nBug distribution:')
print(df['is_buggy'].value_counts())
print(f'\nBug rate: {df["is_buggy"].mean():.1%}')
df.head()

## Step 4 — Exploratory Data Analysis (EDA)

In [ ]:
# 4a. Basic stats
print('=== Dataset Info ===')
df.info()
print('\n=== Missing Values ===')
print(df.isnull().sum())

In [ ]:
# 4b. Class distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Pie chart
counts = df['is_buggy'].value_counts()
axes[0].pie(counts, labels=['Clean', 'Buggy'], autopct='%1.1f%%',
            colors=['#4CAF50', '#F44336'], startangle=90)
axes[0].set_title('Class Distribution')

# Correlation heatmap
corr = df.corr()
sns.heatmap(corr[['is_buggy']].sort_values('is_buggy', ascending=False),
            ax=axes[1], annot=True, fmt='.2f', cmap='RdYlGn_r', vmin=-1, vmax=1)
axes[1].set_title('Feature Correlation with Bug Label')

plt.tight_layout()
plt.show()

In [ ]:
# 4c. Feature distributions by class
important_features = ['lines_added', 'cyclomatic_complexity',
                       'test_coverage_pct', 'prior_bugs_author']

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for i, feat in enumerate(important_features):
    df.groupby('is_buggy')[feat].plot.kde(ax=axes[i], legend=True)
    axes[i].set_title(feat.replace('_', ' ').title())
    axes[i].legend(['Clean', 'Buggy'])

plt.suptitle('Feature Distributions: Clean vs Buggy Commits', y=1.02)
plt.tight_layout()
plt.show()

## Step 5 — Feature Engineering

In [ ]:
def engineer_features(df):
    df = df.copy()
    
    # Churn ratio
    df['churn_ratio'] = df['lines_added'] / (df['lines_deleted'] + 1)
    
    # Complexity per function
    df['complexity_per_func'] = df['cyclomatic_complexity'] / (df['num_functions'] + 1)
    
    # Files touched per line added (scatter score)
    df['scatter_score'] = df['files_changed'] / (df['lines_added'] + 1)
    
    # Risk hour (commits between 22:00–05:00 are riskier)
    df['is_night_commit'] = ((df['commit_hour'] >= 22) | (df['commit_hour'] <= 5)).astype(int)
    
    # Review quality index
    df['review_quality'] = (
        df['num_reviewers'] * 2 + df['review_comments']
    ) / (df['time_to_review_hours'] + 1)
    
    # Test health
    df['test_health'] = (df['test_coverage_pct'] / 100) * np.log1p(df['num_test_cases'])
    
    return df

df_eng = engineer_features(df)
print(f'Features after engineering: {df_eng.shape[1] - 1}')
print('New features:', ['churn_ratio', 'complexity_per_func', 'scatter_score',
                        'is_night_commit', 'review_quality', 'test_health'])

## Step 6 — Preprocessing pipeline

In [ ]:
# Separate features and target
TARGET = 'is_buggy'
FEATURES = [c for c in df_eng.columns if c != TARGET]

X = df_eng[FEATURES]
y = df_eng[TARGET]

print(f'Feature count : {len(FEATURES)}')
print(f'Target classes: {y.value_counts().to_dict()}')

# Build sklearn preprocessing pipeline
preprocessor = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),   # handle any missing values
    ('scaler',  StandardScaler()),                    # normalise for some models
])

print('\n✅ Preprocessing pipeline ready')

## Step 7 — Train / test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y        # keep bug-rate consistent in both splits
)

# Fit + transform
X_train_scaled = preprocessor.fit_transform(X_train)
X_test_scaled  = preprocessor.transform(X_test)

print(f'Train size : {X_train.shape[0]} samples')
print(f'Test  size : {X_test.shape[0]}  samples')
print(f'Bug rate   : train={y_train.mean():.1%}  test={y_test.mean():.1%}')

## Step 8 — Train models

We train two models and pick the best one.

In [ ]:
# --- Model 1: Random Forest ---
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    min_samples_split=5,
    class_weight='balanced',  # handles class imbalance
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train_scaled, y_train)
rf_cv = cross_val_score(rf_model, X_train_scaled, y_train,
                        cv=StratifiedKFold(5), scoring='roc_auc')
print(f'Random Forest  — CV AUC: {rf_cv.mean():.4f} ± {rf_cv.std():.4f}')

In [ ]:
# --- Model 2: XGBoost ---
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

xgb_model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1
)
xgb_model.fit(X_train_scaled, y_train,
              eval_set=[(X_test_scaled, y_test)],
              verbose=False)
xgb_cv = cross_val_score(xgb_model, X_train_scaled, y_train,
                         cv=StratifiedKFold(5), scoring='roc_auc')
print(f'XGBoost        — CV AUC: {xgb_cv.mean():.4f} ± {xgb_cv.std():.4f}')

In [ ]:
# Pick best model
best_model = rf_model if rf_cv.mean() >= xgb_cv.mean() else xgb_model
best_name  = 'Random Forest' if rf_cv.mean() >= xgb_cv.mean() else 'XGBoost'
print(f'\n🏆 Best model selected: {best_name}')

## Step 9 — Evaluate model

In [ ]:
# Predictions
y_pred      = best_model.predict(X_test_scaled)
y_pred_prob = best_model.predict_proba(X_test_scaled)[:, 1]

print(f'=== {best_name} — Test Set Evaluation ===')
print(f'Accuracy : {accuracy_score(y_test, y_pred):.4f}')
print(f'ROC-AUC  : {roc_auc_score(y_test, y_pred_prob):.4f}')
print()
print(classification_report(y_test, y_pred, target_names=['Clean', 'Buggy']))

In [ ]:
# Confusion matrix + ROC curve
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Clean', 'Buggy'],
            yticklabels=['Clean', 'Buggy'], ax=axes[0])
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')
axes[0].set_title('Confusion Matrix')

# ROC curve for both models
for model, name, color in [
    (rf_model,  'Random Forest', '#1976D2'),
    (xgb_model, 'XGBoost',       '#E53935'),
]:
    probs = model.predict_proba(X_test_scaled)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, probs)
    auc = roc_auc_score(y_test, probs)
    axes[1].plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})', color=color, lw=2)

axes[1].plot([0, 1], [0, 1], 'k--', lw=1)
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Feature importance
importances = pd.Series(best_model.feature_importances_, index=FEATURES)
top_features = importances.sort_values(ascending=False).head(15)

plt.figure(figsize=(10, 6))
top_features.plot(kind='barh', color='steelblue')
plt.gca().invert_yaxis()
plt.title(f'Top 15 Feature Importances — {best_name}')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

print('Top 5 bug predictors:')
for feat, imp in top_features.head(5).items():
    print(f'  {feat:<30} {imp:.4f}')

## Step 10 — Save and reload model

In [ ]:
# Save model + preprocessor
MODEL_PATH = 'bug_predictor_model.pkl'
PREP_PATH  = 'bug_predictor_preprocessor.pkl'

joblib.dump(best_model,  MODEL_PATH)
joblib.dump(preprocessor, PREP_PATH)
print(f'✅ Model saved to       {MODEL_PATH}')
print(f'✅ Preprocessor saved to {PREP_PATH}')

# Reload
loaded_model = joblib.load(MODEL_PATH)
loaded_prep  = joblib.load(PREP_PATH)
print('\n✅ Model reloaded successfully!')

# Quick sanity check
sanity_pred = loaded_model.predict(loaded_prep.transform(X_test))
assert (sanity_pred == y_pred).all(), 'Reload sanity check FAILED'
print('✅ Sanity check passed — predictions match!')

## Step 11 — Predict on new data

In [ ]:
def predict_bug_risk(commit_dict, model, prep, feature_names):
    """
    Predict bug probability for a single commit.
    
    commit_dict : dict with raw feature values
    Returns     : dict with prediction and probability
    """
    df_input = pd.DataFrame([commit_dict])
    df_input = engineer_features(df_input)   # apply same feature engineering
    
    # Ensure correct column order and fill missing
    for col in feature_names:
        if col not in df_input.columns:
            df_input[col] = 0
    df_input = df_input[feature_names]
    
    X_scaled = prep.transform(df_input)
    prediction = model.predict(X_scaled)[0]
    probability = model.predict_proba(X_scaled)[0][1]
    
    risk_level = (
        '🟢 LOW'    if probability < 0.3 else
        '🟡 MEDIUM' if probability < 0.6 else
        '🔴 HIGH'
    )
    
    return {
        'is_buggy':    bool(prediction),
        'probability': round(probability, 4),
        'risk_level':  risk_level
    }


# --- Example 1: risky commit ---
risky_commit = {
    'lines_added': 480, 'lines_deleted': 20, 'files_changed': 15,
    'cyclomatic_complexity': 45, 'num_functions': 2, 'avg_function_length': 95,
    'developer_experience': 0, 'prior_bugs_author': 40,
    'commit_hour': 23, 'is_weekend': 1,
    'num_reviewers': 0, 'review_comments': 0, 'time_to_review_hours': 0,
    'test_coverage_pct': 10, 'num_test_cases': 2,
}

# --- Example 2: safe commit ---
safe_commit = {
    'lines_added': 30, 'lines_deleted': 5, 'files_changed': 1,
    'cyclomatic_complexity': 4, 'num_functions': 3, 'avg_function_length': 20,
    'developer_experience': 10, 'prior_bugs_author': 2,
    'commit_hour': 11, 'is_weekend': 0,
    'num_reviewers': 3, 'review_comments': 8, 'time_to_review_hours': 4,
    'test_coverage_pct': 90, 'num_test_cases': 50,
}

print('=== Risky Commit ===')
result = predict_bug_risk(risky_commit, loaded_model, loaded_prep, FEATURES)
print(f'  Bug Predicted : {result["is_buggy"]}')
print(f'  Probability   : {result["probability"]:.1%}')
print(f'  Risk Level    : {result["risk_level"]}')

print('\n=== Safe Commit ===')
result2 = predict_bug_risk(safe_commit, loaded_model, loaded_prep, FEATURES)
print(f'  Bug Predicted : {result2["is_buggy"]}')
print(f'  Probability   : {result2["probability"]:.1%}')
print(f'  Risk Level    : {result2["risk_level"]}')

---
## Summary

| Step | What was done |
|------|---------------|
| 1 | Installed all dependencies |
| 2 | Imported ML / plotting libraries |
| 3 | Loaded commit-level dataset |
| 4 | Explored class balance, correlations, distributions |
| 5 | Engineered 6 new risk-signal features |
| 6 | Built imputer + scaler preprocessing pipeline |
| 7 | Stratified 80/20 train-test split |
| 8 | Trained Random Forest + XGBoost with cross-validation |
| 9 | Evaluated with confusion matrix, ROC-AUC, feature importance |
| 10 | Saved + reloaded model with joblib |
| 11 | Single-commit prediction with risk labels |

**To use on real data:** replace Step 3 with `pd.read_csv('your_file.csv')` and make sure your CSV has the same feature column names.